# Layer-wise linear probe 결과 시각화

BCIC 2B / TUAB / TUEV 세 데이터셋의 layer별 method 비교 그래프.

먼저 아래 **설정 셀**을 한 번 실행한 뒤, 원하는 데이터셋 셀만 실행하면 됩니다 (서로 독립적).

In [ ]:
# ==========================================
# 설정: import, 폰트, 공통 색상 팔레트, 플로팅 함수
# (아래 데이터셋 셀들을 실행하기 전에 먼저 한 번 실행)
# ==========================================
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np

# 한글 subtitle이 깨지지 않도록, 시스템에 설치된 한글 폰트를 자동으로 찾아서 적용한다.
_KOREAN_FONT_CANDIDATES = [
    'NanumGothic', 'Noto Sans CJK KR', 'Noto Sans KR', 'Malgun Gothic',
    'AppleGothic', 'Apple SD Gothic Neo',
]
_installed = {f.name for f in fm.fontManager.ttflist}
_font_found = next((name for name in _KOREAN_FONT_CANDIDATES if name in _installed), None)
if _font_found:
    plt.rcParams['font.family'] = _font_found
else:
    print(
        "[경고] 한글 폰트를 찾지 못했습니다. subtitle의 한글이 네모(tofu)로 깨질 수 있습니다.\n"
        "  Ubuntu/Debian: sudo apt-get install -y fonts-nanum  (설치 후 커널 재시작)"
    )
plt.rcParams['axes.unicode_minus'] = False

# method 이름 -> 색상. 데이터셋마다 등장하는 method가 겹치므로(VERA, Dynamic Pearl 등)
# 모든 그래프에서 같은 method는 항상 같은 색을 쓰도록 여기서 한 번만 정의한다.
# 6개 색 전부가 서로 다른 hue 계열(blue/yellow/teal/red/violet/green)이 되도록 골라서,
# 같은 계열(예: 주황 vs 빨강)끼리 묶이는 조합이 없게 했다. validate_palette.js
# --pairs all 기준(같은 차트에 안 나오는 조합까지 전부)으로 통과 확인:
#   CVD(색맹) 시뮬레이션, 일반 시야 모두 임계값 이상으로 구분됨.
PALETTE = {
    'Linear probe':  '#2A78D6',  # blue
    'VERA':          '#EDA100',  # yellow / gold
    'Pearl':         '#1BAF7A',  # teal green
    'Fine tuning':   '#E34948',  # red
    'Dynamic Pearl': '#4A3AA7',  # violet
    'LORA':          '#008300',  # green
}


def plot_layerwise_metrics(data, metrics, title, subtitle=None, figsize=None, save_path=None):
    """
    layer(1~8)별로 여러 method를 여러 metric에 대해 나란히 line plot으로 그린다.

    data: {method_name: {metric_name: [layer1_val, ..., layerN_val]}}
    metrics: 표시할 metric 이름 리스트 (이 순서대로 subplot이 생성됨)
    title: 그래프 상단 제목 (예: 'TUAB (pooling)')
    subtitle: 제목 아래 한 줄로 들어가는 코멘트 (선택)
    save_path: 지정하면 해당 경로에 PNG로도 저장
    """
    n_layers = len(next(iter(next(iter(data.values())).values())))
    layers = np.arange(1, n_layers + 1)
    n_metrics = len(metrics)

    fig, axes = plt.subplots(1, n_metrics, figsize=figsize or (5.2 * n_metrics, 4.0))
    if n_metrics == 1:
        axes = [axes]

    for ax, metric in zip(axes, metrics):
        for method, method_data in data.items():
            color = PALETTE.get(method, '#666666')
            ax.plot(
                layers, method_data[metric],
                marker='o', markersize=5, linewidth=2,
                color=color, label=method, zorder=3,
            )
        ax.set_title(metric, fontsize=12, color='#333333', pad=10)
        ax.set_xlabel('Layer', fontsize=10, color='#666666')
        ax.set_xticks(layers)
        ax.grid(True, axis='y', linewidth=0.6, color='#e5e5e5', zorder=0)
        for spine in ('top', 'right'):
            ax.spines[spine].set_visible(False)
        for spine in ('left', 'bottom'):
            ax.spines[spine].set_color('#cccccc')
        ax.tick_params(colors='#666666', labelsize=9)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.0),
        ncol=len(data), frameon=False, fontsize=10,
    )

    full_title = title if not subtitle else f"{title}\n{subtitle}"
    fig.suptitle(full_title, fontsize=13, fontweight='bold', y=1.16 if subtitle else 1.1, color='#1a1a1a')

    plt.tight_layout(rect=[0, 0, 1, 0.92 if subtitle else 0.95])
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    return fig

## BCIC 2B (pooling)

In [ ]:
bcic2b_data = {
    'Linear probe': {
        'Balanced accuracy': [50.74, 52.21, 48.38, 49.12, 53.38, 52.5, 53.82, 55],
        "Cohen's kappa":     [1.47, 4.41, -3.24, -1.76, 6.76, 5, 7.65, 10],
        'AUROC':             [49.58, 51.51, 48.43, 51.4, 53.72, 55.31, 55.34, 56.18],
    },
    'VERA': {
        'Balanced accuracy': [51.18, 47.21, 50.44, 51.03, 54.85, 54.85, 56.32, 52.94],
        "Cohen's kappa":     [2.35, -5.59, 0.88, 2.06, 9.71, 9.71, 12.65, 5.88],
        'AUROC':             [49.67, 50.82, 50.71, 53.45, 54.84, 56.59, 56.19, 55.25],
    },
    'Pearl': {
        'Balanced accuracy': [53.68, 53.09, 53.38, 52.65, 53.82, 51.62, 52.5, 54.56],
        "Cohen's kappa":     [7.35, 6.18, 6.7, 5.29, 7.65, 3.24, 5, 9.12],
        'AUROC':             [54.45, 53.83, 54.43, 54.61, 56.52, 55.7, 57.33, 56.05],
    },
}

plot_layerwise_metrics(
    bcic2b_data,
    metrics=['Balanced accuracy', "Cohen's kappa", 'AUROC'],
    title='BCIC 2B (pooling)',
)

## TUAB (pooling)

In [ ]:
tuab_data = {
    'Fine tuning': {
        'Balanced accuracy': [71.91, 72.05, 71.52, 71.42, 72.27, 71.86, 70.54, 69.43],
        "Cohen's kappa":     [44.18, 44.38, 43.52, 43.5, 44.75, 44.17, 41.71, 38.64],
        'AUROC':             [79.5, 79.11, 79.24, 79.71, 79.66, 79.52, 77.5, 76.01],
    },
    'VERA': {
        'Balanced accuracy': [67.85, 71.96, 72.79, 73.32, 73.92, 73.84, 73.9, 73.89],
        "Cohen's kappa":     [35.84, 44.31, 46.07, 46.99, 48, 47.89, 48.3, 47.86],
        'AUROC':             [74.26, 80.04, 81.37, 81.73, 82.21, 81.8, 82.44, 82.3],
    },
    'Dynamic Pearl': {
        'Balanced accuracy': [70.49, 71.88, 73.27, 73.2, 74.25, 73.6, 74.25, 74.67],
        "Cohen's kappa":     [41.42, 44.27, 46.99, 46.86, 48.79, 47.68, 48.92, 49.31],
        'AUROC':             [77.86, 79.85, 80.76, 81.33, 82.2, 82.11, 82.65, 82.45],
    },
}

plot_layerwise_metrics(
    tuab_data,
    metrics=['Balanced accuracy', "Cohen's kappa", 'AUROC'],
    title='TUAB (pooling)',
    subtitle='Pearl 계열이 fine-tuning보다 좋게 나오고 후반 layer에서 많이 좋아져서 아쉬움',
)

## TUEV (pooling)

In [ ]:
tuev_data = {
    'Fine tuning': {
        'Balanced accuracy': [29.85, 31.6, 32.65, 34.15, 34.99, 35.96, 35.24, 39.22],
        "Cohen's kappa":     [32.5, 36.86, 38.49, 39.86, 40.23, 41.96, 40.23, 44.66],
        'Weighted F1':       [66.66, 69.02, 69.67, 70.26, 70.69, 70.77, 70.33, 72.83],
    },
    'LORA': {
        'Balanced accuracy': [30.71, 35.17, 34.89, 34.42, 33.42, 34.2, 38.35, 35.44],
        "Cohen's kappa":     [31.79, 42.01, 41.97, 42.98, 40.88, 41.43, 41.73, 43.75],
        'Weighted F1':       [66.36, 71.1, 70.48, 71.69, 70.19, 70.51, 71.09, 72.28],
    },
    'Dynamic Pearl': {
        'Balanced accuracy': [32.55, 32.07, 32.99, 32.61, 34.65, 33.31, 36.22, 34.48],
        "Cohen's kappa":     [32.17, 38.58, 38.36, 37.62, 40.76, 40.77, 41.46, 44.69],
        'Weighted F1':       [65.69, 69.76, 69.96, 68.06, 70.51, 70.45, 70.96, 72.68],
    },
}

plot_layerwise_metrics(
    tuev_data,
    metrics=['Balanced accuracy', "Cohen's kappa", 'Weighted F1'],
    title='TUEV (pooling)',
    subtitle='Fine tuning이 후반 layer로 갈수록 좋아져서 이론적으로 괜찮아 보이지만 pearl이 첫 layer에서만 좋고 뒤에서는 성능이 많이 뒤쳐져서 아쉬움',
)